# Predict New York City Airbnb Prices
### AIML Minor Project | Linear Regression

**Dataset:** [New York City Airbnb Open Data (AB_NYC_2019)](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data) by AB NYC / Kaggle  
**Expected raw file:** `AB_NYC_2019.csv` (approximately 48,895 listings and 16 columns)

This notebook follows the assignment's **Steps 1 through 7 in the given order**. It contains the data audit, documented cleaning decisions, visual outlier checks, 95th-percentile trimming, categorical encoding, an 80/20 split, a standard `LinearRegression` model, evaluation metrics, and residual interpretation.

### Before running
1. Download `AB_NYC_2019.csv` from the Kaggle dataset page linked above.
2. In Google Colab, run the data-loading cell and upload the CSV when prompted. In local Jupyter, put the CSV in the same folder as this notebook (or change `DATA_FILE`).
3. Run every cell from top to bottom. The metrics and final report text are calculated from the actual run; no model scores are hard-coded.

**Important scope note:** The assignment asks us to remove price and minimum-night outliers using the 95th-percentile rule before splitting. Therefore, the evaluation describes the filtered, typical-market listings, not luxury-price or unusually long-minimum-stay listings that were removed.

## Imports and reproducibility

In [ ]:
# Core data analysis and visualization libraries
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Scikit-learn tools for splitting, modeling, and evaluation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

RANDOM_STATE = 42
TEST_SIZE = 0.20


## Load the dataset

In [ ]:
# Expected filename from the Kaggle dataset.
DATA_FILE = Path("AB_NYC_2019.csv")

# If the file is not already in the working folder, prompt for an upload in Colab.
if not DATA_FILE.exists():
    try:
        from google.colab import files
        print("Upload the original AB_NYC_2019.csv file downloaded from Kaggle.")
        uploaded = files.upload()
        csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("No CSV file was uploaded. Please upload AB_NYC_2019.csv.")
        DATA_FILE = Path(csv_files[0])
    except ImportError as exc:
        raise FileNotFoundError(
            "AB_NYC_2019.csv was not found. Place it beside this notebook or set DATA_FILE to its path."
        ) from exc

df = pd.read_csv(DATA_FILE)

# Basic schema check: fail early if a different file was uploaded by mistake.
required_columns = {
    "id", "name", "host_id", "host_name", "neighbourhood_group",
    "neighbourhood", "latitude", "longitude", "room_type", "price",
    "minimum_nights", "number_of_reviews", "last_review", "reviews_per_month",
    "calculated_host_listings_count", "availability_365"
}
missing_required = required_columns.difference(df.columns)
if missing_required:
    raise ValueError(f"The uploaded file is missing expected columns: {sorted(missing_required)}")

print(f"Loaded file: {DATA_FILE}")
print(f"Raw dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head())

# Step 1: Exploratory Data Analysis (EDA) & Data Auditing

**Task 1.1 - Shape, columns, and data types.** Inspect the raw dataset before changing it.

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

### Task 1.2 - Numerical and categorical variable types

The raw file mixes quantitative values, text categories, identifiers, and a date-like string. The table below classifies the fields by their **meaning**, not just their pandas dtype:

- **Discrete numerical:** counts or whole-number quantities (for example, minimum nights and number of reviews).
- **Continuous numerical:** measurements that can vary along a scale (for example, latitude, longitude, price, and reviews per month).
- **Nominal categorical:** labels with no natural ranking (for example, borough and room type).
- **Ordinal categorical:** categories with a meaningful order. The provided raw schema has no clear ordinal categorical feature.
- `last_review` is a **temporal/date field**, not an ordinary nominal category. `id` and `host_id` are numeric identifiers, not meaningful quantities.

In [ ]:
# Semantic groups for the known AB_NYC_2019 schema.
discrete_numerical = [
    "minimum_nights",
    "number_of_reviews",
    "calculated_host_listings_count",
    "availability_365",
]
continuous_numerical = [
    "latitude",
    "longitude",
    "price",  # target variable; a dollar amount per night
    "reviews_per_month",
]
identifier_numerical = ["id", "host_id"]
nominal_categorical = [
    "name", "host_name", "neighbourhood_group", "neighbourhood", "room_type"
]
ordinal_categorical = []  # No naturally ordered categorical field in this dataset.
temporal_fields = ["last_review"]

classification_rows = []
for group_name, columns in [
    ("Discrete numerical", discrete_numerical),
    ("Continuous numerical", continuous_numerical),
    ("Numeric identifiers (not predictors)", identifier_numerical),
    ("Nominal categorical", nominal_categorical),
    ("Ordinal categorical", ordinal_categorical),
    ("Temporal/date field", temporal_fields),
]:
    for col in columns:
        if col in df.columns:
            classification_rows.append({"Variable type": group_name, "Column": col, "Pandas dtype": str(df[col].dtype)})

classification_table = pd.DataFrame(classification_rows)
display(classification_table)

print("\nNumber of unique values in selected categorical columns:")
for col in ["neighbourhood_group", "neighbourhood", "room_type", "host_name"]:
    if col in df.columns:
        print(f"{col}: {df[col].nunique(dropna=True):,} unique non-null values")

### Task 1.3 - Missing values: counts and percentages

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100),
})
missing_summary = missing_summary[missing_summary["missing_count"] > 0]
missing_summary = missing_summary.sort_values("missing_count", ascending=False)

if missing_summary.empty:
    print("No missing values found in the raw dataset.")
else:
    display(missing_summary.style.format({"missing_percent": "{:.2f}%"}))

# Step 2: Data Cleaning & Handling Missing Values

### Task 2.1 - Decision for `last_review`

**Decision: drop `last_review`.** It is a date stored as text and has a large block of missing values. The missing dates mostly indicate listings with no review history, while `number_of_reviews` and `reviews_per_month` already capture review activity in a simpler numeric form. Engineering a date-recency feature would also require choosing a reference date for this 2019 snapshot. To keep this baseline model focused and reproducible, the date column is removed.

### Task 2.2 - Investigate and fill `reviews_per_month`

Missing review rates are checked against `number_of_reviews` before imputation. A listing with **zero total reviews** has a logical review rate of **0 reviews per month**, so those missing values are filled with zero. If any missing values remain for listings that do have reviews, they are filled with the median for the same room type; a final overall-median fallback protects against an entirely missing room-type group.

In [ ]:
# Work on a copy so the original raw DataFrame remains available for auditing.
df_clean = df.copy()

rpm_missing = df_clean["reviews_per_month"].isna()
missing_with_zero_reviews = (rpm_missing & df_clean["number_of_reviews"].eq(0)).sum()
missing_with_some_reviews = (rpm_missing & df_clean["number_of_reviews"].gt(0)).sum()

print(f"Missing reviews_per_month values: {rpm_missing.sum():,}")
print(f"Of these, listings with zero total reviews: {missing_with_zero_reviews:,}")
print(f"Of these, listings with one or more reviews: {missing_with_some_reviews:,}")

# Logical replacement: no reviews means zero reviews per month.
zero_review_missing = df_clean["reviews_per_month"].isna() & df_clean["number_of_reviews"].eq(0)
df_clean.loc[zero_review_missing, "reviews_per_month"] = 0.0

# If any reviewed listing still has a missing rate, use its room-type median.
remaining_missing = df_clean["reviews_per_month"].isna()
if remaining_missing.any():
    room_type_medians = df_clean.groupby("room_type")["reviews_per_month"].transform("median")
    df_clean.loc[remaining_missing, "reviews_per_month"] = room_type_medians.loc[remaining_missing]

# Last-resort fallback for a room type with no observed median.
if df_clean["reviews_per_month"].isna().any():
    overall_median = df_clean["reviews_per_month"].median()
    df_clean["reviews_per_month"] = df_clean["reviews_per_month"].fillna(overall_median)

print("Remaining missing reviews_per_month values:", df_clean["reviews_per_month"].isna().sum())


### Task 2.3 - Remove irrelevant identifiers and non-predictive text

Drop unique IDs and descriptive text fields that are not part of this assignment's numerical/categorical feature set: `id`, `host_id`, `name`, and `host_name`. The `last_review` date field is also removed based on the decision above. The high-cardinality `neighbourhood` field is retained until Step 4 so its cardinality can be inspected and explained.

In [ ]:
columns_to_drop_step2 = ["id", "host_id", "name", "host_name", "last_review"]
columns_to_drop_step2 = [col for col in columns_to_drop_step2 if col in df_clean.columns]
df_clean = df_clean.drop(columns=columns_to_drop_step2)
print("Dropped in Step 2:", columns_to_drop_step2)

# Confirm missing values in the remaining analysis columns.
remaining_missing = df_clean.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
if remaining_missing.empty:
    print("No missing values remain in the retained columns.")
else:
    print("Remaining missing values:")
    display(remaining_missing)
    # These are retained model fields; remove only rows that still cannot be used.
    before_dropna = len(df_clean)
    df_clean = df_clean.dropna().copy()
    print(f"Rows removed to handle remaining missing values: {before_dropna - len(df_clean):,}")

print("Shape after Step 2:", df_clean.shape)

# Step 3: Outlier Identification & Treatment

### Task 3.1 - Visual inspection of `price` and `minimum_nights`

The boxplots show extreme values, while the histograms show how observations are concentrated. For readability, the histogram x-axes stop at the 99th percentile; the boxplots retain the outlying points. This axis setting is only for visualization and does not remove records.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Price distribution
axes[0, 0].boxplot(df_clean["price"].dropna(), vert=False, showfliers=True)
axes[0, 0].set_title("Price - Boxplot (raw values)")
axes[0, 0].set_xlabel("Nightly price (USD)")

axes[0, 1].hist(df_clean["price"].dropna(), bins=60, edgecolor="black")
axes[0, 1].set_title("Price - Histogram (visual range up to 99th percentile)")
axes[0, 1].set_xlabel("Nightly price (USD)")
axes[0, 1].set_ylabel("Number of listings")
axes[0, 1].set_xlim(left=0, right=df_clean["price"].quantile(0.99))

# Minimum nights distribution
axes[1, 0].boxplot(df_clean["minimum_nights"].dropna(), vert=False, showfliers=True)
axes[1, 0].set_title("Minimum nights - Boxplot (raw values)")
axes[1, 0].set_xlabel("Minimum nights")

axes[1, 1].hist(df_clean["minimum_nights"].dropna(), bins=60, edgecolor="black")
axes[1, 1].set_title("Minimum nights - Histogram (visual range up to 99th percentile)")
axes[1, 1].set_xlabel("Minimum nights")
axes[1, 1].set_ylabel("Number of listings")
axes[1, 1].set_xlim(left=0, right=df_clean["minimum_nights"].quantile(0.99))

plt.tight_layout()
plt.show()

### Task 3.2 - Apply the 95th-percentile trimming rule

The assignment specifies trimming/capping at the 95th percentile and gives luxury nightly prices and 365+ minimum-night rules as examples. This notebook uses **trimming**: rows above the 95th percentile of `price` or `minimum_nights` are removed. Non-positive prices are also removed because a zero/negative nightly price is not a valid price target for this prediction task. The actual cutoffs and number of removed records are printed below.

In [ ]:
rows_before_outlier_trim = len(df_clean)

# A zero or negative nightly price is not a usable business target.
invalid_price_rows = (df_clean["price"] <= 0).sum()
df_clean = df_clean.loc[df_clean["price"] > 0].copy()

# Calculate the 95th-percentile cutoffs on the valid-price population.
price_95 = df_clean["price"].quantile(0.95)
minimum_nights_95 = df_clean["minimum_nights"].quantile(0.95)

print(f"95th percentile price cutoff: ${price_95:,.2f}")
print(f"95th percentile minimum_nights cutoff: {minimum_nights_95:,.2f} nights")
print(f"Invalid non-positive prices removed: {invalid_price_rows:,}")

# Trim high-price and unusually restrictive minimum-stay listings.
keep_mask = (
    (df_clean["price"] <= price_95)
    & (df_clean["minimum_nights"] <= minimum_nights_95)
    & (df_clean["minimum_nights"] >= 1)
)
df_clean = df_clean.loc[keep_mask].copy()

rows_after_outlier_trim = len(df_clean)
rows_removed_total = rows_before_outlier_trim - rows_after_outlier_trim
print(f"Rows before trimming (including invalid prices): {rows_before_outlier_trim:,}")
print(f"Rows after trimming: {rows_after_outlier_trim:,}")
print(f"Rows removed in Step 3: {rows_removed_total:,}")
print(f"Percentage of Step 2 rows retained: {rows_after_outlier_trim / rows_before_outlier_trim:.2%}")

display(df_clean[["price", "minimum_nights"]].describe().T)

# Step 4: Feature Selection & Categorical Encoding

### Task 4.1 - High cardinality and the Curse of Dimensionality

`neighbourhood` identifies very specific local areas and has 200+ distinct labels, while `neighbourhood_group` represents only five borough-level groups. One-hot encoding every neighbourhood would add a large number of mostly sparse columns and parameters. That expands the feature space, can make the model more complex and harder to interpret, and increases the risk of fitting noise in small neighbourhood groups. The broader borough feature captures useful location information with far fewer columns.

**Decision: drop `neighbourhood` and retain `neighbourhood_group`.**

In [ ]:
neighbourhood_count = df_clean["neighbourhood"].nunique(dropna=True)
borough_count = df_clean["neighbourhood_group"].nunique(dropna=True)
room_type_count = df_clean["room_type"].nunique(dropna=True)

print(f"Unique neighbourhoods: {neighbourhood_count:,}")
print(f"Unique neighbourhood groups (boroughs): {borough_count:,}")
print(f"Unique room types: {room_type_count:,}")

# Retain broader borough-level information but remove hyper-specific neighbourhood.
df_model = df_clean.drop(columns=["neighbourhood"]).copy()


### Task 4.2 - One-Hot Encoding with `drop_first=True`

Linear Regression requires numeric inputs. Convert `neighbourhood_group` and `room_type` into dummy variables. `drop_first=True` removes one reference category from each set, avoiding perfect multicollinearity among the complete set of one-hot columns (the Dummy Variable Trap).

In [ ]:
categorical_columns = ["neighbourhood_group", "room_type"]

# With drop_first=True, one category per original field is the reference category.
df_model = pd.get_dummies(
    df_model,
    columns=categorical_columns,
    drop_first=True,
    dtype=int,
)

print("Shape after one-hot encoding:", df_model.shape)
print("Encoded columns:")
print([col for col in df_model.columns if col.startswith("neighbourhood_group_") or col.startswith("room_type_")])

# Confirm that all retained model inputs are numeric.
non_numeric_columns = df_model.select_dtypes(exclude=[np.number]).columns.tolist()
print("Non-numeric columns remaining:", non_numeric_columns)
assert not non_numeric_columns, f"Unexpected non-numeric columns remain: {non_numeric_columns}"

# Step 5: Feature-Target Separation & Validation Split

### Task 5.1 - Separate `X` and `y`

The target is `price` (nightly USD price). All remaining columns are model features.

In [ ]:
X = df_model.drop(columns=["price"])
y = df_model["price"].astype(float)

print("Feature matrix X shape:", X.shape)
print("Target vector y shape:", y.shape)
print("Number of input features:", X.shape[1])

### Task 5.2 - 80% training / 20% testing split

A fixed `random_state=42` makes the split reproducible. The test set is held back for final evaluation and is not used to fit the regression model.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {len(X_train):,} ({len(X_train) / len(X):.0%})")
print(f"Testing rows:  {len(X_test):,} ({len(X_test) / len(X):.0%})")
print("Random state:", RANDOM_STATE)

# Step 6: Model Training & Evaluation

### Task 6.1 and 6.2 - Fit standard Linear Regression and predict test prices

This uses scikit-learn's standard `LinearRegression` estimator. No feature scaling is required for ordinary least squares to produce predictions, and no regularization or alternative model is introduced.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)

print("Model training complete.")
print(f"Number of training observations: {len(X_train):,}")
print(f"Number of test predictions: {len(y_pred):,}")

### Task 6.3 - Evaluation metrics

- **R-squared (R²):** how much of the variation in test-set prices is explained by the model, relative to predicting the test-set mean. A value closer to 1 is stronger; a negative value means the model performs worse than that simple mean baseline on this test set.
- **Mean Absolute Error (MAE):** average absolute prediction error, in US dollars per night. This is the most direct business-facing measure of the typical dollar miss.
- **Mean Squared Error (MSE):** average squared error, in USD²; large misses receive greater weight.
- **Root Mean Squared Error (RMSE):** square root of MSE, in USD per night, making it easier to compare with the price scale.

In [ ]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

metrics_table = pd.DataFrame({
    "Metric": ["R-squared (R²)", "Mean Absolute Error (MAE)", "Mean Squared Error (MSE)", "Root Mean Squared Error (RMSE)"],
    "Value": [r2, mae, mse, rmse],
    "Units / interpretation": ["Unitless", "USD per night", "USD²", "USD per night"],
})
display(metrics_table)

print(f"R²   = {r2:.4f}")
print(f"MAE  = ${mae:,.2f} per night")
print(f"MSE  = {mse:,.2f} USD²")
print(f"RMSE = ${rmse:,.2f} per night")

# Step 7: Residual Analysis & Interpretation

### Task 7.1 - Actual vs. Predicted prices

The diagonal 45-degree line represents perfect predictions: every point on that line has predicted price equal to actual price. Points farther away show larger prediction errors.

In [ ]:
# Convert to NumPy arrays for plotting and residual calculation.
y_test_array = np.asarray(y_test)
y_pred_array = np.asarray(y_pred)

plot_min = min(y_test_array.min(), y_pred_array.min())
plot_max = max(y_test_array.max(), y_pred_array.max())

plt.figure(figsize=(8, 7))
plt.scatter(y_test_array, y_pred_array, alpha=0.35, edgecolors="none")
plt.plot([plot_min, plot_max], [plot_min, plot_max], linestyle="--", linewidth=2, label="Perfect prediction (45° line)")
plt.xlabel("Actual nightly price (USD)")
plt.ylabel("Predicted nightly price (USD)")
plt.title("Actual vs. Predicted Airbnb Prices - Test Set")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Task 7.2 - Residual plot and error distribution

Residual = **actual price - predicted price**. Positive residuals mean the model underpredicted the price; negative residuals mean it overpredicted. A desirable residual plot has errors scattered around zero without a strong pattern.

In [ ]:
residuals = y_test_array - y_pred_array

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred_array, residuals, alpha=0.35, edgecolors="none")
axes[0].axhline(0, color="red", linestyle="--", linewidth=1.5)
axes[0].set_xlabel("Predicted nightly price (USD)")
axes[0].set_ylabel("Residual (actual - predicted, USD)")
axes[0].set_title("Residuals vs. Predicted Prices")
axes[0].grid(alpha=0.25)

axes[1].hist(residuals, bins=50, edgecolor="black")
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("Residual (USD)")
axes[1].set_ylabel("Number of listings")
axes[1].set_title("Distribution of Prediction Errors")
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

print(f"Mean residual (bias): ${residuals.mean():,.2f}")
print(f"Median absolute residual: ${np.median(np.abs(residuals)):,.2f}")

### Task 7.3 - Does the model perform better on cheap or expensive listings?

To answer this using the test set rather than guessing, split actual test prices at their median into lower-priced and higher-priced halves. Compare **MAE** (dollar error) and **MAPE** (percentage error). MAE tells us which group has smaller average dollar misses; MAPE helps account for the fact that the same dollar miss is proportionally larger for a cheaper listing. The printed results determine the conclusion.

In [ ]:
error_analysis = pd.DataFrame({
    "actual": y_test_array,
    "predicted": y_pred_array,
})
error_analysis["absolute_error"] = np.abs(error_analysis["actual"] - error_analysis["predicted"])
error_analysis["squared_error"] = (error_analysis["actual"] - error_analysis["predicted"]) ** 2
error_analysis["absolute_percentage_error"] = (
    error_analysis["absolute_error"] / error_analysis["actual"]
) * 100

test_price_median = error_analysis["actual"].median()
error_analysis["price_segment"] = np.where(
    error_analysis["actual"] <= test_price_median,
    "Lower-priced (at/below test median)",
    "Higher-priced (above test median)",
)

segment_performance = error_analysis.groupby("price_segment").agg(
    listings=("actual", "size"),
    MAE_USD=("absolute_error", "mean"),
    RMSE_USD=("squared_error", lambda values: np.sqrt(values.mean())),
    MAPE_percent=("absolute_percentage_error", "mean"),
).reset_index()

display(segment_performance)

segment_mae = segment_performance.set_index("price_segment")["MAE_USD"]
segment_mape = segment_performance.set_index("price_segment")["MAPE_percent"]
lower_label = "Lower-priced (at/below test median)"
higher_label = "Higher-priced (above test median)"

if segment_mae[lower_label] < segment_mae[higher_label]:
    dollar_conclusion = "Lower-priced listings have the smaller average dollar error (MAE)."
elif segment_mae[lower_label] > segment_mae[higher_label]:
    dollar_conclusion = "Higher-priced listings have the smaller average dollar error (MAE)."
else:
    dollar_conclusion = "Both price segments have the same average dollar error (MAE)."

if segment_mape[lower_label] < segment_mape[higher_label]:
    percentage_conclusion = "Lower-priced listings have the smaller average percentage error (MAPE)."
elif segment_mape[lower_label] > segment_mape[higher_label]:
    percentage_conclusion = "Higher-priced listings have the smaller average percentage error (MAPE)."
else:
    percentage_conclusion = "Both price segments have the same average percentage error (MAPE)."

print("Median actual test price used for the split: $", f"{test_price_median:,.2f}")
print("Dollar-error comparison:", dollar_conclusion)
print("Relative-error comparison:", percentage_conclusion)
print(
    "Interpretation: report both comparisons. Expensive listings can have larger dollar errors simply because their prices are higher; "
    "percentage error helps show whether the miss is also larger relative to the listing's price. A basic linear model may also miss "
    "non-linear price effects and property-level details that are not represented by the available features."
)

## Final business-facing summary (auto-generated from this run)

The cell below writes an editable Markdown summary containing the actual R², MAE, MSE, RMSE, filtered sample size, and the cheap/expensive comparison. Use this generated text to fill the accompanying one-page report. **Only submit numeric results after running all cells with the original CSV.**

In [ ]:
from IPython.display import Markdown, display

if r2 >= 0:
    r2_plain_english = (
        f"An R² of {r2:.4f} means the model explains approximately {r2 * 100:.1f}% "
        "of the variation in test-set prices relative to a baseline that always predicts the test-set mean."
    )
else:
    r2_plain_english = (
        f"An R² of {r2:.4f} is negative, which means the model performs worse on this test set "
        "than a baseline that always predicts the test-set mean."
    )

summary_markdown = f"""# Predict New York City Airbnb Prices - Summary Report

**Model:** Standard Linear Regression  
**Dataset:** AB_NYC_2019.csv (2019 NYC Airbnb listings)  
**Rows used after cleaning and 95th-percentile trimming:** {len(df_model):,}  
**Train/test split:** 80% / 20% (random_state={RANDOM_STATE})

## Test-set results

| Metric | Result | Plain-English meaning |
|---|---:|---|
| R-squared (R²) | {r2:.4f} | {r2_plain_english} |
| MAE | ${mae:,.2f} | Predictions are off by about ${mae:,.2f} per listing per night on average, ignoring direction. |
| MSE | {mse:,.2f} USD² | Squared errors; larger misses receive more weight. |
| RMSE | ${rmse:,.2f} | Typical error scale with extra penalty for large misses. |

## Cheap vs. expensive listings

- {dollar_conclusion}
- {percentage_conclusion}
- The comparison uses the median actual price in the test set (${test_price_median:,.2f}) as the dividing point.

## Key takeaways and limitations

- The model uses location at borough level, room type, geographic coordinates, review activity, minimum nights, host listing count, and availability.
- The neighbourhood field was removed to avoid hundreds of sparse dummy variables; this sacrifices hyper-local pricing detail.
- The 95th-percentile trimming means results apply to the retained typical-market listings, not the excluded extreme-price or long-minimum-stay cases.
- The data is a 2019 snapshot and may not represent today's NYC market. The baseline linear model cannot fully represent non-linear effects or omitted factors such as amenities, seasonality, and exact neighbourhood.

## Final interpretation

{r2_plain_english} MAE of ${mae:,.2f} means the model's absolute prediction error averages about ${mae:,.2f} for each nightly listing price. These are predictive associations, not evidence that any feature causes a price change.
"""

summary_path = Path("NYC_Airbnb_1_Page_Summary_Report.md")
summary_path.write_text(summary_markdown, encoding="utf-8")
display(Markdown(summary_markdown))
print(f"\nSaved report text to: {summary_path.resolve()}")

# In Google Colab, uncomment the next two lines to download the generated summary:
# from google.colab import files
# files.download(str(summary_path))